# 09 — Sandboxed Execution (real OpenEnv, CPU-only)

`OpenEnvHarness` wraps a gym-like OpenEnv `Environment` (`reset`/`step`) so the spine's
conformance/replay/`EventLog` machinery applies to it. This notebook builds a **genuine**
`openenv.core.Environment` subclass — real `openenv` `Observation`/`Action` types and a real
`openenv` `Rubric` computing the reward — and drives it through the full spine.

Honest scope note: there is no natural "external dataset" for a sandboxed-execution harness
demo — the environment here is a real interactive game (number guessing), and what's real is
the library integration itself (genuine `openenv` base classes, not hand-rolled fakes), not an
external corpus. CPU-only, no model, no network — this notebook tests the harness/env contract,
not model quality (that's every other notebook here).

In [1]:
from openenv.core import Environment, Observation, Action, State
from openenv.core.rubrics import Rubric

from agenttune.agentic.harness import run_conformance, replay
from agenttune.agentic.harness_openenv import OpenEnvHarness
from agenttune.agentic.strategy import run_episode, ReActStrategy
from agenttune.agentic.events import Event, EventKind, EventLog

print("[openenv] real openenv.core imports OK -- Environment, Observation, Action, Rubric")

[openenv] real openenv.core imports OK -- Environment, Observation, Action, Rubric


## A genuine OpenEnv `Environment`

Real `Observation`/`Action` Pydantic types, a real `Rubric` computing the reward from the
environment's own terminal condition.

In [2]:
class GuessObs(Observation):
    text: str = ""

class GuessAction(Action):
    n: int = 0

class CorrectGuessRubric(Rubric):
    """Real openenv Rubric: reward 1.0 exactly when the episode terminated on a correct guess."""
    def forward(self, action, observation) -> float:
        return 1.0 if getattr(observation, "done", False) else 0.0

class HigherLowerEnv(Environment):
    """A genuine OpenEnv environment: guess the secret number in [lo, hi]."""
    def __init__(self, secret: int, lo: int = 1, hi: int = 100):
        super().__init__(rubric=CorrectGuessRubric())
        self._secret, self._lo, self._hi = secret, lo, hi
        self._steps = 0

    @property
    def state(self) -> State:
        return State(step_count=self._steps)

    def reset(self, seed=None, episode_id=None, **kwargs) -> GuessObs:
        self._reset_rubric()
        self._steps = 0
        return GuessObs(text=f"I'm thinking of a number between {self._lo} and {self._hi}. Guess it.")

    def step(self, action: GuessAction, timeout_s=None, **kwargs) -> GuessObs:
        self._steps += 1
        guess = int(action.n)
        correct = guess == self._secret
        text = "Correct!" if correct else ("higher" if guess < self._secret else "lower")
        obs = GuessObs(text=text, done=correct)
        obs.reward = self._apply_rubric(action, obs)
        return obs

print("[env] real HigherLowerEnv (openenv.core.Environment subclass) defined")

[env] real HigherLowerEnv (openenv.core.Environment subclass) defined


## 1) One explicit reset/step through `OpenEnvHarness` — the real reward path

In [3]:
LO, HI, SECRET = 1, 100, 73

def build_harness() -> OpenEnvHarness:
    return OpenEnvHarness(
        HigherLowerEnv(SECRET, LO, HI),
        action_space=[{"name": "guess", "arguments": {"n": 0}}],
        action_adapter=lambda a: GuessAction(n=int(a["arguments"]["n"])),
        max_steps=12,
    )

h0 = build_harness()
obs0 = h0.reset("guess the number")
print(f"[reset] obs -> {obs0.text!r}")
o, reward, done, info = h0.step({"name": "guess", "arguments": {"n": SECRET}})
print(f"[step]  guess {SECRET} -> obs={o.text!r} reward={reward} done={done}  (reward from the real Rubric)")

[reset] obs -> "I'm thinking of a number between 1 and 100. Guess it."
[step]  guess 73 -> obs='Correct!' reward=1.0 done=True  (reward from the real Rubric)


## 2) A deterministic binary-search agent plays the real env through `run_episode`

In [4]:
def make_binary_search_policy(lo, hi):
    def policy(state):
        sc = state.scratch
        sc.setdefault("lo", lo)
        sc.setdefault("hi", hi)
        last = state.events[-1].payload.get("text", "") if state.events else ""
        if "higher" in last and "guess" in sc:
            sc["lo"] = sc["guess"] + 1
        elif "lower" in last and "guess" in sc:
            sc["hi"] = sc["guess"] - 1
        sc["guess"] = (sc["lo"] + sc["hi"]) // 2
        return {"name": "guess", "arguments": {"n": sc["guess"]},
               "thought": f"range [{sc['lo']}, {sc['hi']}] -> guess {sc['guess']}"}
    return policy

harness = build_harness()
strategy = ReActStrategy(make_binary_search_policy(LO, HI), max_steps=12)
log = run_episode(strategy, harness, "guess the secret number")
guesses = [e.payload["action"]["arguments"]["n"] for e in log if e.kind is EventKind.TOOL_CALL]
results = [e.payload["output"] for e in log if e.kind is EventKind.TOOL_RESULT]
solved = results[-1] == "Correct!" if results else False
print(f"[episode] agent guesses: {guesses}")
print(f"[episode] solved in {len(guesses)} guesses (log tier={log.tier}, {len(log)} events): {solved}")

[episode] agent guesses: [50, 75, 62, 68, 71, 73]
[episode] solved in 6 guesses (log tier=light, 25 events): True


## 3) Conformance + replay — real harness guarantees over the real env

In [5]:
rep = run_conformance(build_harness())
print(f"[conform] run_conformance -> passed={rep.passed}, drift={rep.drift}")

src = EventLog(events=[
    Event(EventKind.OBSERVATION, {"text": "start"}),
    Event(EventKind.TOOL_CALL, {"action": {"name": "guess", "arguments": {"n": 50}}}),
    Event(EventKind.TOOL_CALL, {"action": {"name": "guess", "arguments": {"n": 73}}}),
])
out = replay(build_harness(), src)
replayed = [e.payload["output"] for e in out if e.kind is EventKind.TOOL_RESULT]
print(f"[replay] re-executed {len(replayed)} tool calls through a fresh real env -> {replayed}")

ok = solved and rep.passed and reward == 1.0 and replayed == ["higher", "Correct!"]
print(f"\n[verdict] real OpenEnv env driven through run_episode + conformance + replay, "
      f"real rubric reward end to end: {ok}")

[conform] run_conformance -> passed=True, drift=[]
[replay] re-executed 2 tool calls through a fresh real env -> ['higher', 'Correct!']

[verdict] real OpenEnv env driven through run_episode + conformance + replay, real rubric reward end to end: True


## Summary

| Check | Real evidence |
|---|---|
| Reward path | real `Rubric.forward` returned `1.0` on the correct guess |
| Episode | binary-search agent solved the real env via `run_episode` |
| Conformance | `run_conformance` passed with no capability drift |
| Replay | a recorded log re-executed deterministically through a fresh real env |

Every `openenv` type above (`Environment`, `Observation`, `Action`, `Rubric`) is the real
library — no hand-rolled fakes — driven through the real `OpenEnvHarness`, CPU-only.